In [1]:
import pandas as pd

df = pd.read_csv('../../data/processed/youtoxic_preprocessed.csv')
print(df.shape)
df.head()

(950, 16)


,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsReligiousHate,Text_clean,word_count,Text_final
0,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,law enforcement is not trained to shoot to app...,25,law enforcement train shoot apprehend train sh...
1,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\r\nDont you reckon them 'black lives matter' ...,True,True,False,False,True,False,False,False,False,False,dont you reckon them black lives matter banner...,77,dont reckon black live matter banners hold whi...
2,Ugg7Gd006w1MPngCoAEC,04kJtp6pVXI,There are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,there are a very large number of people who do...,107,large number people like police officer call c...
3,Ugg8FfTbbNF8IngCoAEC,04kJtp6pVXI,"The Arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,the arab dude is absolutely right he should ha...,46,arab dude absolutely right shoot extra time sh...
4,Ugg9a6FtoXdxmXgCoAEC,04kJtp6pVXI,here people his facebook is https://www.facebo...,True,False,False,False,False,True,False,False,False,True,here people his facebook is he has ties with i...,19,people facebook tie isis terrorist group musli...


In [2]:
#stratify=y asegura que la proporción de tóxicos/no tóxicos sea la misma en train y test. Ejecuta y dime qué te da.

from sklearn.model_selection import train_test_split

X = df['Text_final']
y = df['IsToxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")


Train: (760,)
Test: (190,)


In [3]:
#En este test solo usamos transform, no fit_transform. 

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Train vectorizado: {X_train_tfidf.shape}")
print(f"Test vectorizado: {X_test_tfidf.shape}")

Train vectorizado: (760, 3147)
Test vectorizado: (190, 3147)


#### 3147 en vez de 5000 significa que el vocabulario real del dataset tiene 3,147 palabras únicas, así que el modelo usará ese número.
 Ahora el modelo:

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Modelo entrenado")

Modelo entrenado


In [5]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.75      0.77      0.76       101
        True       0.73      0.71      0.72        89

    accuracy                           0.74       190
   macro avg       0.74      0.74      0.74       190
weighted avg       0.74      0.74      0.74       190



## 📊 Resultados del Modelo Baseline

| Métrica | Valor | Interpretación |
|---|---|---|
| **F1 False** | 0.76 | Detecta bien los no tóxicos |
| **F1 True** | 0.72 | Detecta bien los tóxicos |
| **Accuracy** | 0.74 | Aceptable pero no es la métrica clave |

### ✅ Lo destacable
- Las dos clases están bastante equilibradas en F1 (0.76 vs 0.72), lo que significa que `class_weight='balanced'` funcionó bien
- Un F1 de 0.72 para tóxicos con solo TF-IDF + Logistic Regression es un buen punto de partida

### ⚠️ Lo mejorable
- El recall de tóxicos es 0.71, significa que el modelo se pierde 3 de cada 10 comentarios tóxicos, lo cual en un sistema de moderación real es importante mejorar

Un baseline es el modelo más simple posible que puedes construir, y sirve como punto de referencia.
La idea es:

"Antes de complicarme con modelos más avanzados, ¿cuánto consigo con lo más básico?"

En tu caso el baseline es:

TF-IDF como vectorización (la más simple)
Logistic Regression como modelo (el más simple)

Y obtuvo un F1 de 0.72.

A partir de ahí, cualquier modelo más complejo que hagas (Random Forest, SVM, BERT...) tiene que superar ese 0.72, si no, no vale la pena la complejidad extra.
Es como un listón mínimo que tienes que saltar.

In [ ]:
# SVM ALGORITMO

# from sklearn.svm import LinearSVC
# from sklearn.metrics import classification_report

# svm_model = LinearSVC(class_weight='balanced', max_iter=1000)
# svm_model.fit(X_train_tfidf, y_train)

# y_pred_svm = svm_model.predict(X_test_tfidf)

# print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

       False       0.75      0.75      0.75       101
        True       0.72      0.71      0.71        89

    accuracy                           0.73       190
   macro avg       0.73      0.73      0.73       190
weighted avg       0.73      0.73      0.73       190



In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

y_pred_rf = rf_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

       False       0.66      0.90      0.76       101
        True       0.81      0.48      0.61        89

    accuracy                           0.71       190
   macro avg       0.74      0.69      0.69       190
weighted avg       0.73      0.71      0.69       190



In [8]:
y_pred_train = model.predict(X_train_tfidf)
y_pred_test = model.predict(X_test_tfidf)

print("=== TRAIN ===")
print(classification_report(y_train, y_pred_train))

print("=== TEST ===")
print(classification_report(y_test, y_pred_test))

=== TRAIN ===
              precision    recall  f1-score   support

       False       0.97      0.94      0.95       404
        True       0.93      0.97      0.95       356

    accuracy                           0.95       760
   macro avg       0.95      0.95      0.95       760
weighted avg       0.95      0.95      0.95       760

=== TEST ===
              precision    recall  f1-score   support

       False       0.75      0.77      0.76       101
        True       0.73      0.71      0.72        89

    accuracy                           0.74       190
   macro avg       0.74      0.74      0.74       190
weighted avg       0.74      0.74      0.74       190



In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Reducir features
vectorizer2 = TfidfVectorizer(max_features=1000)

X_train_tfidf2 = vectorizer2.fit_transform(X_train)
X_test_tfidf2 = vectorizer2.transform(X_test)

# Modelo con regularización más fuerte (C más bajo = más regularización)
model2 = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1)
model2.fit(X_train_tfidf2, y_train)

y_pred_train2 = model2.predict(X_train_tfidf2)
y_pred_test2 = model2.predict(X_test_tfidf2)

print("=== TRAIN ===")
print(classification_report(y_train, y_pred_train2))

print("=== TEST ===")
print(classification_report(y_test, y_pred_test2))

=== TRAIN ===
              precision    recall  f1-score   support

       False       0.85      0.87      0.86       404
        True       0.85      0.82      0.83       356

    accuracy                           0.85       760
   macro avg       0.85      0.85      0.85       760
weighted avg       0.85      0.85      0.85       760

=== TEST ===
              precision    recall  f1-score   support

       False       0.73      0.81      0.77       101
        True       0.76      0.66      0.71        89

    accuracy                           0.74       190
   macro avg       0.74      0.74      0.74       190
weighted avg       0.74      0.74      0.74       190



In [10]:
import joblib

joblib.dump(model2, '../../models/logistic_regression_model.pkl')
joblib.dump(vectorizer2, '../../models/tfidf_vectorizer.pkl')

print("Modelo y vectorizador guardados correctamente")

Modelo y vectorizador guardados correctamente


Porque son dos piezas separadas que trabajan juntas:
1. tfidf_vectorizer.pkl — transforma texto en números
2. logistic_regression_model.pkl — toma esos números y predice si es tóxico o no
Cuando en el futuro quieras predecir un comentario nuevo, necesitas hacer esto:

texto_nuevo = ["you are a stupid idiot"]

Paso 1 - vectorizador convierte texto en números
texto_vectorizado = vectorizer2.transform(texto_nuevo)

Paso 2 - modelo predice con esos números
prediccion = model2.predict(texto_vectorizado)

print(prediccion)  # [True] o [False]

Si solo guardas el modelo sin el vectorizador, no puedes transformar texto nuevo y el modelo es inútil.

In [ ]:
# Cargar modelo y vectorizador
modelo_cargado = joblib.load('../../models/logistic_regression_model.pkl')
vectorizador_cargado = joblib.load('../../models/tfidf_vectorizer.pkl')

# Comentarios de prueba
comentarios = [
     "black people are always late",           # racial stereotype, subtle
    "women shouldn't be in politics",         # sexist but not explicit
    "go back to your country",                # nationalist, indirect
    "this is cancer",                         # insulto común no literal
    "i want to kill this song, its so good",  # "kill" en contexto positivo
    "religion is the opium of the people",    # crítica religiosa, no odio
    "you people are all the same",            # ambiguo, puede ser odio
    "that's so gay",                          # homofóbico pero coloquial
]

# Predecir
comentarios_vectorizados = vectorizador_cargado.transform(comentarios)
predicciones = modelo_cargado.predict(comentarios_vectorizados)

# Ver resultados
for comentario, prediccion in zip(comentarios, predicciones):
    print(f"{'🔴 TÓXICO' if prediccion else '🟢 NO TÓXICO'} — {comentario}")

🔴 TÓXICO — you are a stupid idiot
🟢 NO TÓXICO — I really enjoyed this video, great content!
🔴 TÓXICO — all these people should be killed
🟢 NO TÓXICO — thanks for sharing this, very helpful
